# TounsiLM RAG Pipeline — Kaggle Test Notebook

**Features tested:**
- Hybrid retrieval (BM25 + semantic embeddings via RRF)
- Query rewriting with Arabizi normalization
- Automatic query routing by entry type
- Confidence scoring
- Full RAG query (retrieval + TounsiLM-8b generation)

> **Before running:** Enable GPU (T4) and Internet in notebook settings.

## 1 — Environment check

In [ ]:
import torch

print(f"GPU available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2 — Install dependencies

In [ ]:
!pip install -q \
    pydantic>=2.0 \
    chromadb>=0.4.0 \
    sentence-transformers>=2.3.0 \
    transformers>=4.38.0 \
    accelerate>=0.27.0 \
    huggingface_hub>=0.21.0 \
    rank_bm25>=0.2.2 \n    bitsandbytes>=0.43.0

print("Dependencies installed.")

## 3 — Clone the repository

In [ ]:
import os

REPO_URL  = "https://github.com/yasmine-sassi/RAG-pipeline-for-TounsiLM-8b.git"
REPO_NAME = "RAG-pipeline-for-TounsiLM-8b"
REPO_PATH = f"/kaggle/working/{REPO_NAME}"

if os.path.exists(REPO_PATH):
    print("Repo already cloned — pulling latest changes...")
    !git -C "{REPO_PATH}" pull origin main
else:
    !git clone "{REPO_URL}" "{REPO_PATH}"

# Add repo to Python path
import sys
sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)

print(f"Working directory: {os.getcwd()}")

## 4 — HuggingFace authentication

Add your HuggingFace token as a Kaggle secret named `HF_TOKEN`  
*(Notebook settings → Secrets → Add secret)*

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in to HuggingFace.")

## 5 — Build the ChromaDB index

This embeds all 1,647 KB entries with `multilingual-e5-base` and stores them in ChromaDB.  
Run once per session (`--reset` wipes any previous index).

In [ ]:
!python run_rag.py --index --reset

## 6 — Verify the index

In [ ]:
import chromadb
from pathlib import Path

chroma_dir = Path(REPO_PATH) / "rag_kb" / "db" / "chroma_db"
client = chromadb.PersistentClient(path=str(chroma_dir))
collection = client.get_collection("tounsilm_kb")

print(f"Indexed entries: {collection.count()}")

# Show type distribution
from collections import Counter
meta = collection.get(include=["metadatas"])["metadatas"]
counts = Counter(m["type"] for m in meta)
print("\nEntries per type:")
for t, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {t:<15} {n}")

---
## 7 — Query Rewriter

Tests Arabizi normalization and automatic entry-type routing.

In [ ]:
from rag_kb.pipeline.query_rewriter import QueryRewriter

rewriter = QueryRewriter()

test_queries = [
    "شنو معنى برشا؟",
    "shnow maana bar5a?",
    "what is harissa?",
    "shno hia el 7amra?",
    "chno howa el khodra?",
    "شنو هو المثل اللي يقول",
]

print("=" * 60)
for q in test_queries:
    variants = rewriter.rewrite(q)
    routed   = rewriter.detect_type(q)
    print(f"Query   : {q}")
    print(f"Variants: {variants}")
    print(f"Routed  : {routed}")
    print("-" * 60)

---
## 8 — Hybrid retrieval (no LLM)

Tests BM25 + semantic retrieval with RRF fusion.  
Fast to run — does not load TounsiLM-8b.

In [ ]:
from rag_kb.pipeline.retriever import Retriever
from rag_kb.pipeline.query_rewriter import QueryRewriter

retriever = Retriever()
rewriter  = QueryRewriter()

def retrieve_demo(query: str, top_k: int = 5):
    entry_type = rewriter.detect_type(query)
    variants   = rewriter.rewrite(query)

    # Collect best score per doc across all variants
    best = {}
    for v in variants:
        for hit in retriever.retrieve(v, top_k=top_k, entry_type=entry_type):
            doc_id = hit["id"]
            if doc_id not in best or hit["score"] > best[doc_id]["score"]:
                best[doc_id] = hit

    hits    = sorted(best.values(), key=lambda h: h["score"], reverse=True)[:top_k]
    metrics = retriever.calculate_relevance_score(hits)

    print(f"\nQuery     : {query}")
    print(f"Variants  : {variants}")
    print(f"Routed to : {entry_type or 'all types'}")
    print(f"Confidence: {metrics['confidence_level'].upper()}  "
          f"(mean={metrics['mean_score']:.3f}, "
          f"max={metrics['max_score']:.3f}, "
          f"n={metrics['num_results']})")
    print("-" * 60)
    for i, h in enumerate(hits, 1):
        m = h["metadata"]
        print(f"  {i}. [{m.get('type')}] {m.get('term_arabic')} ({m.get('term_arabizi')})  "
              f"score={h['score']:.4f}")
        print(f"     {m.get('meaning', '')[:100]}")
    print("=" * 60)

# --- Run demos ---
retrieve_demo("شنو معنى برشا؟")
retrieve_demo("shnow hia el harissa?")
retrieve_demo("shno maana kh5adra?")          # Arabizi with digit
retrieve_demo("علاش يقولوا ربي يستر؟")        # proverb routing
retrieve_demo("shno maana bibi bel tounsi?")  # code_switch routing

---
## 9 — Full RAG pipeline (retrieval + TounsiLM-8b generation)

Loads TounsiLM-8b — this takes ~2–3 minutes on first run.

In [ ]:
from rag_kb.pipeline.rag_pipeline import RAGPipeline

pipeline = RAGPipeline(top_k=5)
print("Pipeline ready.")

In [ ]:
def rag_demo(question: str, **kwargs):
    result = pipeline.query(
        question=question,
        show_sources=True,
        **kwargs,
    )
    conf_icon = {"high": "🟢", "medium": "🟡", "low": "🔴", "none": "⚫"}.get(
        result["confidence_level"], "?")
    m = result["relevance_metrics"]

    print("\n" + "═" * 65)
    print(f"Question   : {result['question']}")
    print(f"Confidence : {conf_icon} {result['confidence_level'].upper()}  "
          f"(mean={m['mean_score']:.3f}, n={m['num_results']})")
    print("-" * 65)
    print(f"Answer:\n{result['answer']}")
    if result.get("sources"):
        print("\nSources:")
        for h in result["sources"]:
            meta = h["metadata"]
            print(f"  [{h['id']}] {meta.get('term_arabic')} "
                  f"({meta.get('term_arabizi')})  score={h['score']:.3f}")
    print("═" * 65)

### 9a — Expression query

In [ ]:
rag_demo("شنو معنى برشا في الدارجة التونسية؟")

### 9b — Food query (auto-routed)

In [ ]:
rag_demo("شنو هي الهريسة التونسية؟")

### 9c — Arabizi query (tests query rewriter)

In [ ]:
rag_demo("shnow maana el k7li fel tounsi?")

### 9d — Proverb query (auto-routed)

In [ ]:
rag_demo("شنو معنى المثل اللي يقول ربي يستر؟")

### 9e — Color query

In [ ]:
rag_demo("علاش لون سيدي بوسعيد كحلي؟")

### 9f — Low-confidence query (nothing in KB)

In [ ]:
rag_demo("what is the capital of France?")

---
## 10 — Retrieve-only mode (debug retrieval quality)

In [ ]:
!python run_rag.py --retrieve "شنو معنى برشا؟" --top-k 5

In [ ]:
!python run_rag.py --retrieve "shnow hia el harissa" --top-k 3 --type food

---
## 11 — BM25 vs Semantic comparison

Shows the contribution of each retrieval method.

In [ ]:
from rag_kb.pipeline.retriever import Retriever, _SEMANTIC_W, _BM25_W

retriever = Retriever()
query = "barcha"

sem_hits  = retriever._semantic_retrieve(query, k=5, entry_type=None)
bm25_hits = retriever._bm25_retrieve(query, k=5, entry_type=None)
rrf_hits  = retriever.retrieve(query, top_k=5)

def show(label, hits):
    print(f"\n{'─'*20} {label} {'─'*20}")
    for i, h in enumerate(hits[:5], 1):
        m = h['metadata']
        print(f"  {i}. {m.get('term_arabic'):15} ({m.get('term_arabizi'):15})  score={h['score']:.4f}")

show(f"Semantic only  (w={_SEMANTIC_W})", sem_hits)
show(f"BM25 only      (w={_BM25_W})", bm25_hits)
show("Hybrid RRF (final)", rrf_hits)

---
## 12 — Custom query

Change `MY_QUERY` and run to test any question.

In [ ]:
MY_QUERY = "شنو معنى والاه في الدارجة؟"

rag_demo(MY_QUERY)